# Projekt 02 (medium): Statistik aus dem Computer — Bootstrap & Permutationstest

**Szenario:** Dein Team hat einen A/B-Test gefahren: Variante B der Produktseite
gegen die bisherige Variante A, je 4.000 Besucher. Zwei Fragen an dich als Analyst:

1. Kaufen mit B **mehr** Besucher? (Konversionsrate)
2. Geben Kaeufer mit B **mehr Geld** aus? (Bestellwert)

Du beantwortest beides ohne einzige Verteilungsformel — mit **Bootstrap**
(Konfidenzintervalle) und **Permutationstest** (p-Werte), beides selbst gebaut.

**Daten: synthetisch, mit bekannter Wahrheit.** Die Daten werden in der ersten
Zelle mit festem Seed erzeugt. Der Clou: **Wir kennen die wahren Effekte** (steht
am Ende des Notebooks!) und koennen pruefen, ob unsere Methoden sie finden.
Genau das kann kein echter Datensatz bieten — deshalb hier synthetisch.

**Bezug zum Skript:** Abschnitte 3.1 (A/B-Design) und 3.2 (Bootstrap, Permutation),
plus Modul 02 Abschnitt 3.1 (p-Werte, Konfidenzintervalle).

## 1. Die Daten

- `konv_a`, `konv_b`: pro Besucher True/False — hat er/sie gekauft?
- `wert_a`, `wert_b`: Bestellwerte der Kaeufer in EUR (nur Kaeufer!)

Schau dir die Daten kurz an (EDA-Reflex aus Modul 02): Wie viele Kaeufe, welche
Raten, wie sehen die Bestellwerte aus?

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)   # Daten-Seed — NICHT aendern, sonst stimmen die Checks nicht

N_A = N_B = 4000
konv_a = rng.random(N_A) < 0.048          # (die wahren Raten stehen am Notebook-Ende)
konv_b = rng.random(N_B) < 0.052
wert_a = rng.lognormal(np.log(50), 0.6, konv_a.sum())
wert_b = rng.lognormal(np.log(56), 0.6, konv_b.sum())

print(f"A: {konv_a.sum()} Kaeufe von {N_A} ({konv_a.mean():.2%})")
print(f"B: {konv_b.sum()} Kaeufe von {N_B} ({konv_b.mean():.2%})")
print(f"Median Bestellwert: A {np.median(wert_a):.2f} EUR, B {np.median(wert_b):.2f} EUR")

fig, achsen = plt.subplots(1, 2, figsize=(11, 3.5), sharex=True)
for achse, werte, name in [(achsen[0], wert_a, "A"), (achsen[1], wert_b, "B")]:
    achse.hist(werte, bins=30, edgecolor="black")
    achse.axvline(np.median(werte), color="red", label=f"Median {np.median(werte):.0f}")
    achse.set_title(f"Bestellwerte Variante {name}"); achse.legend()
plt.show()

**Befund:** Rechtsschiefe Bestellwerte (typisch fuer Geldbetraege!) → der **Median**
ist die richtige Kennzahl (Modul 02, Skript 1.3). Aber: Fuer den Standardfehler des
Medians gibt es keine so bequeme Formel wie $s/\sqrt{n}$ fuer den Mittelwert.
Zeit fuer den Bootstrap.

## 2. Bootstrap: Wie unsicher ist unser Median?

**Idee (Skript 3.2):** Wir haben nur EINE Stichprobe. Der Bootstrap tut so, als
waere sie die ganze Population: Wir ziehen aus ihr viele neue Stichproben gleicher
Groesse **mit Zuruecklegen** und berechnen jedes Mal den Median. Die Streuung dieser
Bootstrap-Mediane zeigt, wie stark der Median von Stichprobe zu Stichprobe schwankt.

**Aufgabe:** Implementiere `bootstrap_verteilung(werte, kennzahl, n_boot)`:
`n_boot`-mal eine Stichprobe der Laenge `len(werte)` mit Zuruecklegen ziehen
(`rng_boot.choice(werte, size=len(werte))` — `replace=True` ist Standard) und
`kennzahl` darauf anwenden. Rueckgabe als numpy-Array.

In [ ]:
rng_boot = np.random.default_rng(1)   # eigener Seed fuer die Resampling-Verfahren

def bootstrap_verteilung(werte, kennzahl, n_boot=10_000):
    """Bootstrap: n_boot Kennzahlen aus Stichproben mit Zuruecklegen."""
    # TODO: n_boot-mal ziehen und kennzahl(...) sammeln (siehe Aufgabentext)

boot_medians_b = bootstrap_verteilung(wert_b, np.median)
ki_low, ki_high = np.percentile(boot_medians_b, [2.5, 97.5])

plt.hist(boot_medians_b, bins=50, edgecolor="black")
plt.axvline(ki_low, color="red"); plt.axvline(ki_high, color="red")
plt.title(f"Bootstrap-Verteilung des Medians (B) — 95%-KI: [{ki_low:.2f}, {ki_high:.2f}]")
plt.xlabel("Median (EUR)"); plt.show()

# Mini-Check: das KI sollte grob im Bereich [52, 69] liegen und ~10-16 EUR breit sein
print(50 < ki_low < 58 and 63 < ki_high < 72)

**Lesart:** „Der Median von B liegt bei ~61 EUR; plausibel sind (95 %-KI) Werte
zwischen ~53 und ~68 EUR." Die Unsicherheit ist betraechtlich — wir haben eben nur
~200 Kaeufe. Ohne Bootstrap haette man dem Punktwert 61,21 vermutlich blind geglaubt.

**Aufgabe (Transfer):** Berechne das 95%-Bootstrap-KI fuer die **Differenz der
Mediane** (B − A). Tipp: In jeder Bootstrap-Runde aus *beiden* Gruppen ziehen und
die Median-Differenz speichern. Liegt die 0 im Intervall?

In [ ]:
# TODO: eigene Funktion — pro Runde aus a UND b ziehen, Median-Differenz sammeln
# TODO: lo, hi = np.percentile(..., [2.5, 97.5])
print(f"beobachtete Differenz: {np.median(wert_b) - np.median(wert_a):.2f} EUR")
print(f"95%-KI der Differenz: [{lo:.2f}, {hi:.2f}] — enthaelt 0: {lo <= 0 <= hi}")

## 3. Permutationstest: Koennte das Zufall sein?

Das KI der Differenz deutet schon an, dass B besser ist. Jetzt der formale Test.

**Idee (Skript 3.2):** Unter der Nullhypothese „B wirkt nicht" ist die Gruppenzugehoerigkeit
bedeutungslos — dann duerfte man die Labels beliebig mischen. Also: alle Werte in
einen Topf, zufaellig neu auf zwei Gruppen verteilen, Differenz berechnen — 10.000-mal.
Der **p-Wert** ist der Anteil der gemischten Differenzen, die (im Betrag) mindestens
so gross sind wie die beobachtete.

**Aufgabe:** Implementiere `permutationstest(a, b, kennzahl)` fuer die Differenz
`kennzahl(b_gemischt) - kennzahl(a_gemischt)` und wende ihn auf die **Bestellwert-Mediane** an.

In [ ]:
def permutationstest(a, b, kennzahl, n_perm=10_000):
    """Liefert (beobachtete_differenz, p_wert, null_verteilung)."""
    beobachtet = kennzahl(b) - kennzahl(a)
    # TODO: Topf bilden (concatenate), n_perm-mal permutieren,
    # TODO: Differenz der Kennzahl sammeln, p = Anteil |null| >= |beobachtet|
    return beobachtet, p, null

beob, p, null = permutationstest(wert_a, wert_b, np.median)
plt.hist(null, bins=50, edgecolor="black")
plt.axvline(beob, color="red", linewidth=2, label=f"beobachtet: {beob:.2f}")
plt.title(f"Nullverteilung der Median-Differenz — p = {p:.4f}")
plt.xlabel("Median-Differenz unter H0 (EUR)"); plt.legend(); plt.show()

# Mini-Check: klar signifikant
print(p < 0.02)

**Lesart:** Wuerde B nichts bewirken, waere eine Median-Differenz von ~12 EUR
extrem unwahrscheinlich (p ≈ 0,005). **Kaeufer geben mit B signifikant mehr aus.**

**Aufgabe:** Jetzt dieselbe Maschinerie fuer die **Konversionsrate**:
`permutationstest(konv_a, konv_b, np.mean)` (der Mittelwert von True/False-Werten
ist die Rate!). Was kommt heraus?

In [ ]:
# TODO: Permutationstest auf konv_a / konv_b mit np.mean
print(f"Differenz der Konversionsraten: {beob_k:.4f} ({beob_k:.2%}-Punkte)")
print(f"p-Wert: {p_k:.3f}")

# Mini-Check: NICHT signifikant
print(p_k > 0.05)

## 4. Die Aufloesung — und die wichtigste Lektion

Zeit, die **wahren** Parameter aus der Datenzelle zu verraten:

| Groesse | Wahrheit (Generator) | Test-Ergebnis |
|--|--|--|
| Konversionsrate | A: 4,8 %, B: 5,2 % — **B ist wirklich besser!** | p ≈ 0,18 → nicht signifikant |
| Median Bestellwert | A: 50 EUR, B: 56 EUR — B wirklich besser | p ≈ 0,005 → signifikant ✓ |

Der Konversions-Effekt **existiert**, aber der Test hat ihn **nicht gefunden**:
Bei Raten um 5 % und einem Unterschied von 0,4 Prozentpunkten reichen 2×4.000
Besucher schlicht nicht (zu wenig **Power**, Skript 3.1 — solche Effekte brauchen
Zehntausende Besucher pro Gruppe).

**Die Lektionen:**
1. **„Nicht signifikant" heisst NICHT „kein Effekt"** — oft heisst es nur „zu wenig Daten".
2. Deshalb macht man die Power-Analyse **vor** dem Test: Haette man ausgerechnet,
   dass 2×4.000 Besucher fuer den erwarteten Effekt nie reichen, haette man sich
   diesen Teil des Tests sparen (oder laenger laufen lassen) koennen.
3. Und umgekehrt: Der beobachtete Median-Unterschied (12,29 EUR) **ueberschaetzt**
   den wahren (6 EUR) — auch signifikante Effekte sind mit Rauschen behaftet;
   darum immer das Konfidenzintervall mit berichten (der wahre Wert 6 liegt drin? pruefe!).

**Aufgabe zum Abschluss:** Formuliere in 3 Saetzen eine Empfehlung ans Produktteam.
(Musterantwort in der Loesung.)

<details><summary>Musterempfehlung</summary>

Variante B erhoeht den Bestellwert der Kaeufer deutlich (Median +12 EUR beobachtet,
95%-KI ca. +3 bis +21 EUR) — das ist statistisch belastbar. Ob B auch die Konversionsrate
verbessert, koennen wir mit diesem Stichprobenumfang nicht beurteilen (p = 0,18 bei
sehr kleinem erwartbarem Effekt); dafuer muesste der Test mit deutlich mehr Besuchern
bzw. laengerer Laufzeit wiederholt werden. Empfehlung: B ausrollen (Bestellwert-Gewinn
ist gesichert) und den Konversionseffekt in einem groesser dimensionierten Folgetest messen.
</details>

## Geschafft — was du jetzt kannst

- Bootstrap-Konfidenzintervalle fuer beliebige Kennzahlen bauen (auch Differenzen)
- Permutationstests implementieren und p-Werte *wirklich* verstehen
  (du hast die Nullverteilung mit eigenen Augen gesehen)
- Power ernst nehmen: nicht signifikant ≠ kein Effekt
- Ergebnisse als Handlungsempfehlung formulieren

**Bonusaufgaben:**
1. Wiederhole den Konversions-Permutationstest mit 2×40.000 Besuchern (Datenzelle
   kopieren und anpassen) — wird der Effekt jetzt gefunden?
2. Vergleiche dein Bootstrap-KI fuer den **Mittelwert** von `wert_b` mit der
   Formel-Variante $\bar{x} \pm 1{,}96 \cdot s/\sqrt{n}$ — wie nah liegen sie beieinander?
3. `scipy.stats.mannwhitneyu(wert_a, wert_b)` ist ein klassischer Rang-Test fuer
   dieselbe Frage — vergleiche den p-Wert mit deinem Permutationstest.